In [1]:
import numpy as np
import pandas as pd
from typing import List, Optional
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score,
                             average_precision_score,
                            brier_score_loss,
                            precision_recall_curve,
                            classification_report,
                            confusion_matrix)
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier
import joblib
import matplotlib.pyplot as plt

In [2]:
RANDOM_STATE = 42
 
RAW_COLS = [
    "id", "loan_status",
    "annual_inc", "emp_length", "home_ownership", "verification_status",
    "fico_range_low", "fico_range_high", "earliest_cr_line",
    "open_acc", "total_acc", "delinq_2yrs", "pub_rec",
    "dti", "revol_bal", "revol_util",
    "loan_amnt", "term", "int_rate", "installment",
    "purpose", "grade", "sub_grade",
    "issue_d", "addr_state"
]
 
STATUS_MAP = {
    "Fully Paid": 0,
    "Charged Off": 1,
    "Default": 1,
    "Does not meet the credit policy. Status: Fully Paid": 0,
    "Does not meet the credit policy. Status: Charged Off": 1,
}

DATA_PATH = "/Users/alex._choo/Desktop/CAP5771/Loan DS/accepted_2007_to_2018Q4.csv/accepted_2007_to_2018Q4.csv"

chunk_iter = pd.read_csv(
    DATA_PATH,
    usecols=RAW_COLS,
    chunksize=200_000,
    low_memory=False
)
 
clean_chunks = []
dropped_unmapped = 0
 
for chunk in chunk_iter:
    chunk.columns = chunk.columns.str.strip()
 
    # Rename id -> loan_id
    if "id" in chunk.columns:
        chunk = chunk.rename(columns={"id": "loan_id"})
 
    # Normalize status text
    chunk["loan_status"] = (
        chunk["loan_status"]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.replace(r"Status:\s*", "Status: ", regex=True)
    )
 
    # Build binary target
    chunk["target"] = chunk["loan_status"].map(STATUS_MAP)
    dropped_unmapped += int(chunk["target"].isna().sum())
 
    # Keep only mapped statuses
    chunk = chunk[chunk["target"].notna()].copy()
    chunk["target"] = chunk["target"].astype("int8")
 
    # Prevent leakage (loan_status directly encodes target)
    chunk = chunk.drop(columns=["loan_status"])
 
    clean_chunks.append(chunk)
 
df_clean = pd.concat(clean_chunks, ignore_index=True)
 
# One row per loan_id
df_clean = df_clean.drop_duplicates(subset=["loan_id"]).reset_index(drop=True)
 
# Safety checks
assert "loan_status" not in df_clean.columns
assert set(df_clean["target"].unique()) <= {0, 1}
 
print("df_clean shape:", df_clean.shape)
print("Dropped unmapped status rows:", dropped_unmapped)
print("Target distribution:\n", df_clean["target"].value_counts(dropna=False))

df_clean shape: (1348099, 25)
Dropped unmapped status rows: 912602
Target distribution:
 target
0    1078739
1     269360
Name: count, dtype: int64


In [19]:
display(df_clean.head())
display(df_clean.describe().T)

,loan_id,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,...,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,open_acc,pub_rec,revol_bal,revol_util,total_acc,target
0,70686,5000.0,36 months,7.75,156.11,A,A3,10+ years,MORTGAGE,70000.0,...,NaN,NaN,770.0,774.0,NaN,NaN,0.0,NaN,NaN,0
1,87023,7500.0,36 months,13.75,255.43,E,E2,< 1 year,OWN,22000.0,...,1.0,Oct-2003,660.0,664.0,7.0,0.0,4175.0,51.5,8.0,0
2,88637,6000.0,36 months,10.59,195.28,C,C2,< 1 year,RENT,20000.0,...,0.0,Jan-1996,695.0,699.0,5.0,0.0,13660.0,66.0,6.0,0
3,88046,4400.0,36 months,9.64,141.25,B,B4,2 years,MORTGAGE,30000.0,...,0.0,Jul-2004,695.0,699.0,4.0,0.0,3493.0,63.5,5.0,0
4,85961,1200.0,36 months,9.01,38.17,B,B2,< 1 year,RENT,36000.0,...,NaN,NaN,705.0,709.0,NaN,NaN,0.0,NaN,NaN,0


,count,mean,min,25%,50%,75%,max,std
loan_amnt,1348099.0,14408.998913,500.0,7975.0,12000.0,20000.0,40000.0,8716.137925
int_rate,1348099.0,13.241562,5.31,9.75,12.74,15.99,30.99,4.765685
installment,1348099.0,437.777843,4.93,248.28,375.04,580.22,1719.83,261.49719
annual_inc,1348095.0,76237.743295,0.0,45750.0,65000.0,90000.0,10999200.0,69922.741975
issue_d,1348099,2015-06-02 04:04:23.848871168,2007-06-01 00:00:00,2014-07-01 00:00:00,2015-08-01 00:00:00,2016-07-01 00:00:00,2018-12-01 00:00:00,NaN
dti,1347725.0,18.274253,-1.0,11.79,17.61,24.05,999.0,11.155495
delinq_2yrs,1348070.0,0.317633,0.0,0.0,0.0,0.0,39.0,0.877744
fico_range_low,1348099.0,696.162233,610.0,670.0,690.0,710.0,845.0,31.850787
fico_range_high,1348099.0,700.162371,614.0,674.0,694.0,714.0,850.0,31.851434
open_acc,1348070.0,11.590447,0.0,8.0,11.0,14.0,90.0,5.47468


In [4]:
_MISSING_TOKENS = {"", "na", "n/a", "none", "null", "nan"}

def _normalize_missing_str(s: pd.Series) -> pd.Series:
    s = s.astype("string").str.strip()
    low = s.str.lower()
    is_missing = s.isna() | low.isin(_MISSING_TOKENS)
    return s.mask(is_missing, pd.NA)

def _to_datetime_strict(s: pd.Series, date_format: str) -> pd.Series:
    if pd.api.types.is_datetime64_any_dtype(s):
        return s
    s_clean = _normalize_missing_str(s)
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    mask = s_clean.notna()
    if mask.any():
        # strict: raises if invalid non-missing date exists
        out.loc[mask] = pd.to_datetime(s_clean.loc[mask], format=date_format)
    return out

def _to_numeric_strict(s: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s)

    s_clean = _normalize_missing_str(s).str.replace(",", "", regex=False)
    out = pd.Series(np.nan, index=s.index, dtype="float64")
    mask = s_clean.notna()
    if mask.any():
        # strict: raises if invalid non-missing numeric exists
        out.loc[mask] = pd.to_numeric(s_clean.loc[mask])
    return out


# Parse Dates
class ParseDatesTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, date_cols: List[str], date_format: str = "%b-%Y"):
        self.date_cols = date_cols
        self.date_format = date_format

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for c in self.date_cols:
            if c in X.columns:
                X[c] = _to_datetime_strict(X[c], self.date_format)
        return X


class DateFeaturesTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, date_col="issue_d", date_format: str = "%b-%Y"):
        self.date_col = date_col
        self.date_format = date_format

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if self.date_col in X.columns:
            X[self.date_col] = _to_datetime_strict(X[self.date_col], self.date_format)
            X[self.date_col + "_month"] = X[self.date_col].dt.month
            X[self.date_col + "_weekday"] = X[self.date_col].dt.weekday
            X[self.date_col + "_quarter"] = X[self.date_col].dt.quarter
            X[self.date_col + "_year"] = X[self.date_col].dt.year
            X[self.date_col + "_weekofyear"] = X[self.date_col].dt.isocalendar().week.astype("float64")
        
        return X


# Engineer FICO + Credit Age
class FicoAndCreditAgeTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, date_format: str = "%b-%Y"):
        self.date_format = date_format

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        if "fico_range_low" in X.columns and "fico_range_high" in X.columns:
            X["fico_range_low"] = _to_numeric_strict(X["fico_range_low"])
            X["fico_range_high"] = _to_numeric_strict(X["fico_range_high"])
            X["fico"] = (X["fico_range_low"] + X["fico_range_high"]) / 2

        if "issue_d" in X.columns and "earliest_cr_line" in X.columns:
            X["issue_d"] = _to_datetime_strict(X["issue_d"], self.date_format)
            X["earliest_cr_line"] = _to_datetime_strict(X["earliest_cr_line"], self.date_format)

            credit_age = (X["issue_d"] - X["earliest_cr_line"]).dt.days / 365.25
            X["credit_age_years"] = credit_age.where(credit_age >= 0, np.nan)

        return X

# Clean Categorical Variables
class CategoricalCleaner(BaseEstimator, TransformerMixin):
    def __init__(self, cols: List[str], rare_thresh=0.01):
        self.cols = cols
        self.rare_thresh = rare_thresh
        self.keep_categories_ = {}
 
    def fit(self, X, y=None):
        for col in self.cols:
            if col in X.columns:
                s = _normalize_missing_str(X[col]).str.lower()
                vc = s.value_counts(normalize=True, dropna=True)
                top = vc[vc >= self.rare_thresh].index.tolist()
                self.keep_categories_[col] = set(top)
        return self
 
    def transform(self, X):
        X = X.copy()
        for col in self.cols:
            if col in X.columns:
                s = _normalize_missing_str(X[col]).str.lower()
                allowed = self.keep_categories_.get(col, None)
                if allowed is not None:
                    s = s.where(s.isin(allowed) | s.isna(), other="__other__")
                X[col] = s
        return X

class GradeOrdinalTransformer(BaseEstimator, TransformerMixin):
    """
    Map grade A..G -> 1..7 and sub_grade like 'A1'->11, 'B3'->23 etc.
    Keeps NaN as NaN.
    """
    def __init__(self, grade_col="grade", sub_col="sub_grade"):
        self.grade_col = grade_col
        self.sub_col = sub_col
        self.grade_map = {g: i for i, g in enumerate(list("ABCDEFG"), start=1)}

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        if self.grade_col in X.columns:
            g = X[self.grade_col].astype("string").str.strip().str.upper()
            X[self.grade_col + "_ord"] = g.map(self.grade_map)

        if self.sub_col in X.columns:
            s = X[self.sub_col].astype("string").str.strip().str.upper()

            def sub_map(v):
                if pd.isna(v):
                    return np.nan
                v = str(v).strip().upper()
                if len(v) < 2:
                    return np.nan
                letter = v[0]
                if letter not in self.grade_map:
                    return np.nan
                num = "".join([c for c in v[1:] if c.isdigit()])
                if not num:
                    return np.nan
                n = int(num)
                if n < 1 or n > 5:
                    return np.nan
                return self.grade_map[letter] * 10 + n

            X[self.sub_col + "_ord"] = s.map(sub_map)

        return X

class PercentToFloatTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, cols: List[str]):
        self.cols = cols
 
    def fit(self, X, y=None):
        return self
 
    def transform(self, X):
        X = X.copy()
        for c in self.cols:
            if c in X.columns:
                s = _normalize_missing_str(X[c])
                s = s.str.replace("%", "", regex=False).str.replace(",", "", regex=False).str.strip()
                X[c] = _to_numeric_strict(s)
 
                if c == "revol_util":
                    X[c] = X[c].where(X[c] >= 0, np.nan)
                    X[c] = X[c].clip(lower=0, upper=100)
        return X
    
class TermExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, col="term"):
        self.col = col
 
    def fit(self, X, y=None):
        return self
 
    def transform(self, X):
        X = X.copy()
        if self.col in X.columns:
            s = _normalize_missing_str(X[self.col])
            s = s.str.extract(r"(\d+)")[0]
            X[self.col] = _to_numeric_strict(s)
        return X
 
 
class EmpLengthTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, col="emp_length"):
        self.col = col
 
    def fit(self, X, y=None):
        return self
 
    def transform(self, X):
        X = X.copy()
        if self.col in X.columns:
            s = _normalize_missing_str(X[self.col]).str.lower()
            s = s.str.replace(r"10\+", "10", regex=True)
            s = s.str.replace(r"<\s*1", "0", regex=True)
            nums = s.str.extract(r"(\d+)", expand=False)
            X[self.col + "_years"] = _to_numeric_strict(nums)
        return X
    
class DropColumnsTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, cols: List[str]):
        self.cols = cols
 
    def fit(self, X, y=None):
        return self
 
    def transform(self, X):
        X = X.copy()
        existing = [c for c in self.cols if c in X.columns]
        return X.drop(columns=existing)
    
class DerivedFeaturesTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
 
    def transform(self, X):
        X = X.copy()
 
        for c in ["revol_bal", "loan_amnt", "installment", "annual_inc", "open_acc", "total_acc", "revol_util", "fico"]:
            if c in X.columns:
                X[c] = _to_numeric_strict(X[c])
 
        if "revol_bal" in X.columns and "loan_amnt" in X.columns:
            denom = X["loan_amnt"].replace({0: np.nan})
            X["revol_to_loan"] = X["revol_bal"] / denom
 
        if "installment" in X.columns and "annual_inc" in X.columns:
            monthly = (X["annual_inc"] / 12).replace({0: np.nan})
            X["installment_to_monthly_income"] = X["installment"] / monthly
 
        if "open_acc" in X.columns and "total_acc" in X.columns:
            denom = X["total_acc"].replace({0: np.nan})
            X["open_to_total_ratio"] = X["open_acc"] / denom
 
        if "credit_age_years" in X.columns:
            X["credit_age_bucket"] = pd.cut(
                X["credit_age_years"],
                bins=[-1, 1, 3, 5, 10, 20, 100],
                labels=False
            )
 
        if "fico" in X.columns and "revol_util" in X.columns:
            X["fico_times_revolutil"] = X["fico"] * (X["revol_util"] / 100.0)
 
        return X

class LogTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, cols):
        self.cols = cols
 
    def fit(self, X, y=None):
        return self
 
    def transform(self, X):
        X = X.copy()
        for c in self.cols:
            if c in X.columns:
                vals = _to_numeric_strict(X[c])
                X[c] = np.log1p(vals)
        return X
 
 
class NumericWinsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, cols: List[str], lower_q: float = 0.01, upper_q: float = 0.99):
        self.cols = cols
        self.lower_q = lower_q
        self.upper_q = upper_q
        self.bounds_ = {}
 
    def fit(self, X, y=None):
        for col in self.cols:
            if col in X.columns:
                s = _to_numeric_strict(X[col])
                lo = s.quantile(self.lower_q)
                hi = s.quantile(self.upper_q)
                if pd.notna(lo) and pd.notna(hi):
                    self.bounds_[col] = (lo, hi)
        return self
 
    def transform(self, X):
        X = X.copy()
        for col, (lo, hi) in self.bounds_.items():
            if col in X.columns:
                X[col] = _to_numeric_strict(X[col]).clip(lower=lo, upper=hi)
                X[col] = X[col].where(X[col] >= 0, np.nan)
        return X
 
 
class MissingIndicatorAndMedianImputer(BaseEstimator, TransformerMixin):
    def __init__(self, numeric_cols: List[str]):
        self.numeric_cols = numeric_cols
        self.medians_ = {}
 
    def fit(self, X, y=None):
        for col in self.numeric_cols:
            if col in X.columns:
                self.medians_[col] = _to_numeric_strict(X[col]).median()
        return self
 
    def transform(self, X):
        X = X.copy()
        for col, med in self.medians_.items():
            if col in X.columns:
                vals = _to_numeric_strict(X[col])
                X[col + "_missing"] = vals.isna().astype(int)
                X[col] = vals.fillna(med)
        return X

In [5]:
def build_cleaning_pipeline(
    numeric_cols: List[str],
    categorical_cols: List[str],
    log_cols: Optional[List[str]] = None,
    include_grade_ord: bool = True
):
    steps = []
 
    # dedupe should be done before split/pipeline to avoid X/y mismatch
    steps.append(("parse_dates", ParseDatesTransformer(
        date_cols=["issue_d", "earliest_cr_line"],
        date_format="%b-%Y"
    )))
    steps.append(("date_features", DateFeaturesTransformer(
        date_col="issue_d",
        date_format="%b-%Y"
    )))
    steps.append(("percent_to_float", PercentToFloatTransformer(cols=["int_rate", "revol_util"])))
    steps.append(("term_extract", TermExtractor(col="term")))
    steps.append(("emp_len", EmpLengthTransformer(col="emp_length")))
    steps.append(("fico_credit", FicoAndCreditAgeTransformer(date_format="%b-%Y")))
    steps.append(("drop_raw_fico", DropColumnsTransformer(cols=["fico_range_low", "fico_range_high"])))
 
    if include_grade_ord:
        steps.append(("grade_ord", GradeOrdinalTransformer(grade_col="grade", sub_col="sub_grade")))
 
    steps.append(("cat_clean", CategoricalCleaner(cols=categorical_cols, rare_thresh=0.01)))
    steps.append(("derived", DerivedFeaturesTransformer()))
 
    if log_cols is not None and len(log_cols) > 0:
        steps.append(("log_transform", LogTransformer(cols=log_cols)))
 
    derived_num_cols = [
        "revol_to_loan",
        "installment_to_monthly_income",
        "open_to_total_ratio",
        "fico_times_revolutil"
    ]
    all_num_cols = list(dict.fromkeys(numeric_cols + derived_num_cols))
 
    steps.append(("winsorize", NumericWinsorizer(
        cols=all_num_cols,
        lower_q=0.01,
        upper_q=0.99
    )))
    steps.append(("impute", MissingIndicatorAndMedianImputer(
        numeric_cols=all_num_cols
    )))
 
    # keep engineered date parts, drop raw ids/labels/raw dates
    steps.append(("drop_non_feature_cols", DropColumnsTransformer(
        cols=["loan_id", "loan_status", "target", "issue_d", "earliest_cr_line"]
    )))
 
    return Pipeline(steps)

In [6]:
# Column groups
numeric_cols_fund = [
    "annual_inc", "open_acc", "total_acc", "delinq_2yrs", "pub_rec", "dti",
    "revol_bal", "revol_util", "loan_amnt",
    "fico", "credit_age_years", "emp_length_years",
    "revol_to_loan", "open_to_total_ratio", "fico_times_revolutil"
]
categorical_cols_fund = ["home_ownership", "verification_status", "purpose", "addr_state"]
 
numeric_cols_full = numeric_cols_fund + ["int_rate", "installment"]
categorical_cols_full = categorical_cols_fund + ["grade", "sub_grade"]
 
log_cols = ["annual_inc", "revol_bal", "loan_amnt"]

In [7]:
# Strict date parse for split
df_clean = df_clean.copy()
df_clean["issue_d"] = _to_datetime_strict(df_clean["issue_d"], "%b-%Y")
 
if df_clean["issue_d"].isna().any():
    n_bad = int(df_clean["issue_d"].isna().sum())
    raise ValueError(f"issue_d has {n_bad} missing/invalid values after strict parsing.")
 
df_clean = df_clean.sort_values("issue_d").reset_index(drop=True)
 
train_df = df_clean[df_clean["issue_d"].dt.year <= 2015].copy()
val_df   = df_clean[df_clean["issue_d"].dt.year == 2016].copy()
test_df  = df_clean[df_clean["issue_d"].dt.year >= 2017].copy()
 
print("rows total / train / val / test:", len(df_clean), len(train_df), len(val_df), len(test_df))

rows total / train / val / test: 1348099 829355 293105 225639


In [8]:
# Fundamental / full frames
fund_drop = ["grade", "sub_grade", "int_rate", "installment"]
missing_drop = [c for c in fund_drop if c not in df_clean.columns]
if missing_drop:
    raise KeyError(f"Missing required columns for fundamental drop: {missing_drop}")
 
train_fund = train_df.drop(columns=fund_drop).copy()
val_fund   = val_df.drop(columns=fund_drop).copy()
test_fund  = test_df.drop(columns=fund_drop).copy()
 
train_full = train_df.copy()
val_full   = val_df.copy()
test_full  = test_df.copy()

In [9]:
# Targets
for name, d in {
    "train_fund": train_fund, "val_fund": val_fund, "test_fund": test_fund,
    "train_full": train_full, "val_full": val_full, "test_full": test_full
}.items():
    if "target" not in d.columns:
        raise KeyError(f"'target' missing in {name}")
 
y_train_f = train_fund["target"].astype(int).copy()
y_val_f   = val_fund["target"].astype(int).copy()
y_test_f  = test_fund["target"].astype(int).copy()
 
y_train_full = train_full["target"].astype(int).copy()
y_val_full   = val_full["target"].astype(int).copy()
y_test_full  = test_full["target"].astype(int).copy()

In [10]:
# Drop leakage/id from X
for d in (train_fund, val_fund, test_fund, train_full, val_full, test_full):
    d.drop(columns=["target"], inplace=True)
    if "loan_id" in d.columns:
        d.drop(columns=["loan_id"], inplace=True)

In [11]:
# Build pipelines
pipeline_logistic = build_cleaning_pipeline(
    numeric_cols=numeric_cols_fund,
    categorical_cols=categorical_cols_fund,
    log_cols=log_cols,
    include_grade_ord=False
)
 
pipeline_xgb = build_cleaning_pipeline(
    numeric_cols=numeric_cols_fund,
    categorical_cols=categorical_cols_fund,
    log_cols=None,
    include_grade_ord=False
)
 
pipeline_xgb_full = build_cleaning_pipeline(
    numeric_cols=numeric_cols_full,
    categorical_cols=categorical_cols_full,
    log_cols=None,
    include_grade_ord=True
)

In [12]:
# Fit + transform pipelines
# pipeline_logistic.fit(train_fund)
# pipeline_xgb.fit(train_fund)
# pipeline_xgb_full.fit(train_full)

# logistic pipeline with log transforms
X_train_fund_clean = pipeline_logistic.fit_transform(train_fund)
X_val_fund_clean   = pipeline_logistic.transform(val_fund)
X_test_fund_clean  = pipeline_logistic.transform(test_fund)
 
# xgb pipeline with log 
X_train_fund_xgb = pipeline_xgb.fit_transform(train_fund)
X_val_fund_xgb   = pipeline_xgb.transform(val_fund)
X_test_fund_xgb  = pipeline_xgb.transform(test_fund)

# xgb cleaning pipeline (no log transforms)
X_train_full_xgb = pipeline_xgb_full.fit_transform(train_full)
X_val_full_xgb   = pipeline_xgb_full.transform(val_full)
X_test_full_xgb  = pipeline_xgb_full.transform(test_full)

/Users/alex._choo/anaconda3/envs/am126/lib/python3.13/site-packages/sklearn/pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/alex._choo/anaconda3/envs/am126/lib/python3.13/site-packages/sklearn/pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/alex._choo/anaconda3/envs/am126/lib/python3.13/site-packages/sklearn/pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/alex._cho

In [20]:
display(X_train_fund_clean.head())
display(X_train_fund_clean.describe().T)

,loan_amnt,term,emp_length,home_ownership,annual_inc,verification_status,purpose,addr_state,dti,delinq_2yrs,...,dti_missing,revol_bal_missing,revol_util_missing,loan_amnt_missing,fico_missing,credit_age_years_missing,emp_length_years_missing,revol_to_loan_missing,open_to_total_ratio_missing,fico_times_revolutil_missing
0,8.517393,36.0,10+ years,mortgage,11.156265,not verified,other,ct,8.81,0.0,...,0,0,1,0,0,1,0,0,1,1
1,8.922792,36.0,< 1 year,own,9.998843,not verified,debt_consolidation,ma,14.29,1.0,...,0,0,0,0,0,0,0,0,0,0
2,8.699681,36.0,< 1 year,rent,9.903538,not verified,debt_consolidation,ct,12.90,0.0,...,0,0,0,0,0,0,0,0,0,0
3,8.389587,36.0,2 years,mortgage,10.308986,not verified,debt_consolidation,nj,3.72,0.0,...,0,0,0,0,0,0,0,0,0,0
4,7.496097,36.0,< 1 year,rent,10.491302,not verified,other,tx,3.27,0.0,...,0,0,1,0,0,1,0,0,1,1


,count,mean,std,min,25%,50%,75%,max
loan_amnt,829355.0,9.381752,0.661271,7.496097,8.987322,9.392745,9.903538,10.463132
term,829355.0,42.028772,10.408869,36.000000,36.000000,36.000000,60.000000,60.000000
annual_inc,829355.0,11.066213,0.512887,9.817330,10.714440,11.066654,11.407576,12.429220
dti,829355.0,17.951601,8.210165,1.920000,11.760000,17.480000,23.730000,37.300000
delinq_2yrs,829355.0,0.293137,0.718973,0.000000,0.000000,0.000000,0.000000,4.000000
open_acc,829355.0,11.441582,5.083817,3.000000,8.000000,11.000000,14.000000,28.000000
pub_rec,829355.0,0.179875,0.444959,0.000000,0.000000,0.000000,0.000000,2.000000
revol_bal,829355.0,9.274587,0.976224,5.668292,8.746398,9.357121,9.919558,11.428346
revol_util,829355.0,54.696036,23.763690,2.100000,37.200000,55.600000,73.200000,98.400000
total_acc,829355.0,25.154972,11.564291,6.000000,17.000000,24.000000,32.000000,60.000000


In [13]:
# 1) shapes align
assert X_train_fund_clean.shape[0] == len(y_train_f)
assert X_val_fund_clean.shape[0] == len(y_val_f)
assert X_test_fund_clean.shape[0] == len(y_test_f)

# 2) no leakage columns
for df in [X_train_fund_clean, X_val_fund_clean, X_test_fund_clean]:
    assert "loan_status" not in df.columns
    assert "target" not in df.columns

# 3) missing indicators present for key features
for col in ["emp_length_years_missing", "annual_inc_missing", "revol_util_missing"]:
    print(col, col in X_train_fund_clean.columns)

# 4) medians & bounds learned
print("winsor bounds sample keys:", list(pipeline_logistic.named_steps['winsorize'].bounds_.keys())[:10])
print("median sample:", list(pipeline_logistic.named_steps['impute'].medians_.items())[:10])

emp_length_years_missing True
annual_inc_missing True
revol_util_missing True
winsor bounds sample keys: ['annual_inc', 'open_acc', 'total_acc', 'delinq_2yrs', 'pub_rec', 'dti', 'revol_bal', 'revol_util', 'loan_amnt', 'fico']
median sample: [('annual_inc', 11.06665398721974), ('open_acc', 11.0), ('total_acc', 24.0), ('delinq_2yrs', 0.0), ('pub_rec', 0.0), ('dti', 17.48), ('revol_bal', 9.357121103184378), ('revol_util', 55.6), ('loan_amnt', 9.392745258631441), ('fico', 692.0)]


In [16]:
def encode_categoricals(X_train, X_val, X_test, scale_numeric: bool):
    X_train = X_train.copy()
    X_val = X_val.copy()
    X_test = X_test.copy()

    cat_cols = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    num_cols = [c for c in X_train.columns if c not in cat_cols]

    # normalize categorical missing values to np.nan (avoid pd.NA issues in OHE)
    for d in (X_train, X_val, X_test):
        for c in cat_cols:
            col = d[c].astype("object")
            d[c] = col.where(pd.notna(col), np.nan)

    if scale_numeric:
        num_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ])
    else:
        num_pipe = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median"))
        ])

    cat_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="__missing__")),
        ("ohe", OneHotEncoder(handle_unknown="ignore"))
    ])

    encoder = ColumnTransformer(
        transformers=[
            ("num", num_pipe, num_cols),
            ("cat", cat_pipe, cat_cols),
        ],
        remainder="drop"
    )

    X_train_enc = encoder.fit_transform(X_train)
    X_val_enc = encoder.transform(X_val)
    X_test_enc = encoder.transform(X_test)

    return encoder, X_train_enc, X_val_enc, X_test_enc, num_cols, cat_cols

# Logistic (fundamental): with scaling
enc_fund_log, X_train_fund_clean_enc, X_val_fund_clean_enc, X_test_fund_clean_enc, num_f_log, cat_f_log = encode_categoricals(
    X_train_fund_clean, X_val_fund_clean, X_test_fund_clean, scale_numeric=True
)

# XGB (fundamental): no scaling needed
enc_fund_xgb, X_train_fund_xgb_enc, X_val_fund_xgb_enc, X_test_fund_xgb_enc, num_f_xgb, cat_f_xgb = encode_categoricals(
    X_train_fund_xgb, X_val_fund_xgb, X_test_fund_xgb, scale_numeric=False
)

# XGB (full): no scaling needed
enc_full_xgb, X_train_full_xgb_enc, X_val_full_xgb_enc, X_test_full_xgb_enc, num_full_xgb, cat_full_xgb = encode_categoricals(
    X_train_full_xgb, X_val_full_xgb, X_test_full_xgb, scale_numeric=False
)

print("fund_log cats:", len(cat_f_log), "shape:", X_train_fund_clean_enc.shape)
print("fund_xgb cats:", len(cat_f_xgb), "shape:", X_train_fund_xgb_enc.shape)
print("full_xgb cats:", len(cat_full_xgb), "shape:", X_train_full_xgb_enc.shape)

fund_log cats: 5 shape: (829355, 94)
fund_xgb cats: 5 shape: (829355, 94)
full_xgb cats: 7 shape: (829355, 135)
